Applying flat-table proxy AQPs (e.g., BlazeIt, ABae) to graphs is theoretically intractable due to #P-hard structural enumeration. To provide a rigorous baseline, we use`PROXY-CASCADE-FILTER`: it hard-prunes vertices with proxy scores <0.4, accepts >0.6, and exhausts the identical budget B invoking the Oracle on uncertain vertices in [0.4, 0.6] before tree sampling. Despite strict budget parity, this baseline suffers sever systematic bias (ARE=37.6% on Amazon SUM, vs Proxy's 8.67%). Hard-pruning inevitably produces false negatives, permanently eliminating massive intersecting subgraphs. This proves our soft Importance Sampling is necessary. (Note: Systems like SUPG/ScaleDoc target approximate selection/retrieval rather than unbiased aggregation, serving only as heuristic proxy-acceleration references here).

In [1]:
import json
import numpy as np
import pandas as pd

# ==========================================
# 1. 文件路径配置（按需修改路径）
# ==========================================
est_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/basic_estimates.json"
gt_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json"

# ==========================================
# 2. 读取 JSON 数据
# ==========================================
with open(est_file, "r", encoding="utf-8") as f:
    est_data = json.load(f)

with open(gt_file, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# ==========================================
# 3. 数据对齐与误差计算
# ==========================================
records = []
epsilon = 1e-9  # 防除零保护

for key, t_hat in est_data.items():
    # 统一清洗 key（去除可能存在的 .graph 后缀，防止对齐失败）
    clean_key = str(key).replace(".graph", "")
    
    # 寻找 ground truth 中对应的真实值
    t_true = gt_data.get(key)
    if t_true is None:
        t_true = gt_data.get(f"{clean_key}.graph")
    if t_true is None:
        t_true = gt_data.get(clean_key)
        
    # 过滤无效值（仅对比两边都有且 T_true > 0 的查询）
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed Relative Error): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (Absolute Relative Error, ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": clean_key,
            "T_hat (估计值)": t_hat,
            "T_true (真实值)": t_true,
            "Signed_RE (带符号误差)": signed_re,
            "ARE (绝对误差)": are
        })

df = pd.DataFrame(records)

# ==========================================
# 4. 汇总统计与输出
# ==========================================
if df.empty:
    print("❌ 未成功匹配到任何查询，请检查两个 JSON 文件中的 key 名称是否一致！")
else:
    mean_signed_re = df["Signed_RE (带符号误差)"].mean()
    mean_are = df["ARE (绝对误差)"].mean()
    median_are = df["ARE (绝对误差)"].median()

    print("=" * 65)
    print(f"📊 评估结果汇总 (成功对齐 {len(df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print(f"3. 绝对值相对误差中位数 (Median ARE)    : {median_are:.4f}  ({median_are * 100:.2f}%)")
    print("=" * 65)
    
    # 格式化百分比显示详细表格
    df_display = df.copy()
    df_display["Signed_RE (带符号误差)"] = df_display["Signed_RE (带符号误差)"].map("{:.2%}".format)
    df_display["ARE (绝对误差)"] = df_display["ARE (绝对误差)"].map("{:.2%}".format)
    
    # 在 Jupyter 中完美展示 DataFrame 表格
    display(df_display.head(10))  # 默认展示前 10 条

📊 评估结果汇总 (成功对齐 177 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : 0.2253  (22.53%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.6530  (65.30%)
3. 绝对值相对误差中位数 (Median ARE)    : 0.5409  (54.09%)


,query,T_hat (估计值),T_true (真实值),Signed_RE (带符号误差),ARE (绝对误差)
0,query_3_120,25372.1,19458.59,30.39%,30.39%
1,query_3_122,25007.3,18684.51,33.84%,33.84%
2,query_3_126,25091.9,18684.51,34.29%,34.29%
3,query_3_130,25060.2,18684.51,34.12%,34.12%
4,query_3_133,25482.3,19458.59,30.96%,30.96%
5,query_3_137,24492.6,18296.02,33.87%,33.87%
6,query_3_17,24487.6,18296.02,33.84%,33.84%
7,query_3_21,25985.1,20168.85,28.84%,28.84%
8,query_3_25,24899.9,18684.51,33.26%,33.26%
9,query_3_36,25115.2,18684.51,34.42%,34.42%


In [2]:
import json
import numpy as np
import pandas as pd

# ==========================================
# 1. 文件路径配置（按需修改路径）
# ==========================================
est_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/basic_estimates.json"
gt_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json"

# ==========================================
# 2. 读取 JSON 数据
# ==========================================
with open(est_file, "r", encoding="utf-8") as f:
    est_data = json.load(f)

with open(gt_file, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# ==========================================
# 3. 数据对齐与误差计算
# ==========================================
records = []
epsilon = 1e-9  # 防除零保护

for key, t_hat in est_data.items():
    # 统一清洗 key（去除可能存在的 .graph 后缀，防止对齐失败）
    clean_key = str(key).replace(".graph", "")
    
    # 寻找 ground truth 中对应的真实值
    t_true = gt_data.get(key)
    if t_true is None:
        t_true = gt_data.get(f"{clean_key}.graph")
    if t_true is None:
        t_true = gt_data.get(clean_key)
        
    # 过滤无效值（仅对比两边都有且 T_true > 0 的查询）
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed Relative Error): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (Absolute Relative Error, ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": clean_key,
            "T_hat (估计值)": t_hat,
            "T_true (真实值)": t_true,
            "Signed_RE (带符号误差)": signed_re,
            "ARE (绝对误差)": are
        })

df = pd.DataFrame(records)

# ==========================================
# 4. 汇总统计与输出
# ==========================================
if df.empty:
    print("❌ 未成功匹配到任何查询，请检查两个 JSON 文件中的 key 名称是否一致！")
else:
    mean_signed_re = df["Signed_RE (带符号误差)"].mean()
    mean_are = df["ARE (绝对误差)"].mean()
    
    # 各种分位数计算 (P50/中位数, P70, P90, P95)
    p50_are = df["ARE (绝对误差)"].median()  # 等价于 .quantile(0.50)
    p70_are = df["ARE (绝对误差)"].quantile(0.70)
    p90_are = df["ARE (绝对误差)"].quantile(0.90)
    p95_are = df["ARE (绝对误差)"].quantile(0.95)

    print("=" * 65)
    print(f"📊 评估结果汇总 (成功对齐 {len(df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)
    
    # 格式化百分比显示详细表格
    df_display = df.copy()
    df_display["Signed_RE (带符号误差)"] = df_display["Signed_RE (带符号误差)"].map("{:.2%}".format)
    df_display["ARE (绝对误差)"] = df_display["ARE (绝对误差)"].map("{:.2%}".format)
    
    # 在 Jupyter 中展示前 10 条对比结果
    display(df_display.head(10))

📊 评估结果汇总 (成功对齐 177 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : 0.2253  (22.53%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.6530  (65.30%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.5409  (54.09%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.7126  (71.26%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.9654  (96.54%)
6. 绝对值相对误差 P95 (P95 ARE)         : 1.1253  (112.53%)


,query,T_hat (估计值),T_true (真实值),Signed_RE (带符号误差),ARE (绝对误差)
0,query_3_120,25372.1,19458.59,30.39%,30.39%
1,query_3_122,25007.3,18684.51,33.84%,33.84%
2,query_3_126,25091.9,18684.51,34.29%,34.29%
3,query_3_130,25060.2,18684.51,34.12%,34.12%
4,query_3_133,25482.3,19458.59,30.96%,30.96%
5,query_3_137,24492.6,18296.02,33.87%,33.87%
6,query_3_17,24487.6,18296.02,33.84%,33.84%
7,query_3_21,25985.1,20168.85,28.84%,28.84%
8,query_3_25,24899.9,18684.51,33.26%,33.26%
9,query_3_36,25115.2,18684.51,34.42%,34.42%


In [6]:
import os
import csv
import pandas as pd

dataset_dir = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend"
mapping_csv = os.path.join(dataset_dir, "data_graph", "id_mapping.csv")
product_csv = os.path.join(dataset_dir, "csv_data", "product.csv")
review_csv = os.path.join(dataset_dir, "csv_data", "review.csv")

# 1. 检查 ID Mapping
orig_to_internal = {}
if os.path.exists(mapping_csv):
    with open(mapping_csv, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) >= 2:
                orig_to_internal[row[1].strip()] = int(row[0].strip())

print(f"[*] id_mapping.csv 中共加载到 {len(orig_to_internal)} 条节点映射\n")

# 2. 通用概率分布分析函数
def analyze_table(table_name, csv_path, proxy_col, oracle_col):
    print("=" * 65)
    print(f" 📊 正在分析数据表: {table_name}.csv")
    print("=" * 65)
    
    if not os.path.exists(csv_path):
        print(f"[Error] 找不到文件: {csv_path}\n")
        return
        
    df = pd.read_csv(csv_path)
    p_ids = df["id:ID"].astype(str).str.strip().tolist()
    matched = sum(1 for pid in p_ids if pid in orig_to_internal)
    
    print(f"数据总行数        : {len(df)} 行")
    print(f"成功匹配 ID 映射 : {matched} / {len(df)} (匹配率: {matched/len(df)*100:.2f}%)\n")

    # 依次分析 Proxy 列和 Oracle 列
    for col_name, col_type in [(proxy_col, "Proxy (代理模型)"), (oracle_col, "Oracle (真值/高精度模型)")]:
        if col_name and col_name in df.columns:
            s = df[col_name].dropna()
            print(f"--- 【{col_type} 列】: {col_name} ---")
            print(f"  最小值: {s.min():.4f} | 最大值: {s.max():.4f} | 均值: {s.mean():.4f} | 中位数: {s.median():.4f}")
            
            # 打印分位数，看分布倾斜情况
            q_vals = [round(q, 4) for q in s.quantile([0.25, 0.50, 0.75, 0.90]).tolist()]
            print(f"  分位数 [P25, P50, P75, P90]: {q_vals}")
            
            # 严格按照 <0.4, [0.4, 0.6], >0.6 统计
            hard_reject = (s < 0.4).sum()
            uncertain = ((s >= 0.4) & (s <= 0.6)).sum()
            hard_accept = (s > 0.6).sum()
            total = len(s)
            
            print(f"  硬剔除区域 (< 0.4)    : {hard_reject:7d} 个节点 ({hard_reject/total*100:6.2f}%)")
            print(f"  灰色地带   [0.4, 0.6] : {uncertain:7d} 个节点 ({uncertain/total*100:6.2f}%)")
            print(f"  硬接受区域 (> 0.6)    : {hard_accept:7d} 个节点 ({hard_accept/total*100:6.2f}%)\n")
        else:
            print(f"[Warn] 未在 {table_name}.csv 中找到列名: {col_name}\n")

# 3. 分析 Product 表 (表1)
analyze_table(
    table_name="Product", 
    csv_path=product_csv, 
    proxy_col="ML3_proxy2_probability", 
    oracle_col="ML3_oracle2_probability"
)

# 4. 分析 Review 表 (表2)
analyze_table(
    table_name="Review", 
    csv_path=review_csv, 
    proxy_col="ML2_proxy2_probability", 
    oracle_col="ML2_oracle2_probability"
)

[*] id_mapping.csv 中共加载到 315526 条节点映射

 📊 正在分析数据表: Product.csv
数据总行数        : 5000 行
成功匹配 ID 映射 : 5000 / 5000 (匹配率: 100.00%)

--- 【Proxy (代理模型) 列】: ML3_proxy2_probability ---
  最小值: 0.0000 | 最大值: 1.0000 | 均值: 0.1841 | 中位数: 0.0287
  分位数 [P25, P50, P75, P90]: [0.0049, 0.0287, 0.1725, 0.8476]
  硬剔除区域 (< 0.4)    :    4140 个节点 ( 82.80%)
  灰色地带   [0.4, 0.6] :     163 个节点 (  3.26%)
  硬接受区域 (> 0.6)    :     697 个节点 ( 13.94%)

--- 【Oracle (真值/高精度模型) 列】: ML3_oracle2_probability ---
  最小值: 0.0000 | 最大值: 1.0000 | 均值: 0.1831 | 中位数: 0.0251
  分位数 [P25, P50, P75, P90]: [0.0045, 0.0251, 0.1654, 0.8579]
  硬剔除区域 (< 0.4)    :    4148 个节点 ( 82.96%)
  灰色地带   [0.4, 0.6] :     163 个节点 (  3.26%)
  硬接受区域 (> 0.6)    :     689 个节点 ( 13.78%)

 📊 正在分析数据表: Review.csv
数据总行数        : 214501 行
成功匹配 ID 映射 : 214501 / 214501 (匹配率: 100.00%)

--- 【Proxy (代理模型) 列】: ML2_proxy2_probability ---
  最小值: 0.0006 | 最大值: 1.0000 | 均值: 0.6996 | 中位数: 0.9961
  分位数 [P25, P50, P75, P90]: [0.1597, 0.9961, 1.0, 1.0]
  硬剔除区域 (< 0.4)    :   60

In [ ]:
import os
import csv
import random
import subprocess
import re
import argparse
import json

def load_query_budgets(csv_path, target_frac=0.1, target_method="8_POSSA"):
    """从消融实验 CSV 中读取每个查询在 0.1 采样率下的专属 oracle_cost"""
    query_budgets = {}
    if not os.path.exists(csv_path):
        print(f"[Error] 找不到消融 CSV 文件: {csv_path}")
        return query_budgets

    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                frac = float(row.get('budget_frac', 0))
                if abs(frac - target_frac) > 1e-4: 
                    continue
                
                method = row.get('method', '').strip()
                if method not in ['POSS', '8_POSSA'] and target_method in ['POSS', '8_POSSA']: 
                    continue
                if method != target_method and target_method not in ['POSS', '8_POSSA']: 
                    continue
                
                q_name = row['query_basename'].strip()
                cost = int(float(row['oracle_cost']))
                
                if q_name not in query_budgets:
                    query_budgets[q_name] = []
                query_budgets[q_name].append(cost)
            except Exception:
                continue
                
    # 计算多轮 run_id 的平均预算并四舍五入
    final_budgets = {q: int(round(sum(c)/len(c))) for q, c in query_budgets.items()}
    return final_budgets

def load_id_mapping(mapping_csv):
    """读取 id_mapping.csv, 返回 orig_id -> internal_id"""
    orig_to_internal = {}
    if not os.path.exists(mapping_csv):
        print(f"[Error] 找不到 ID Mapping 文件: {mapping_csv}")
        return orig_to_internal

    with open(mapping_csv, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None) # 跳过表头
        for row in reader:
            if len(row) >= 2:
                try: 
                    orig_to_internal[row[1].strip()] = int(row[0].strip())
                except ValueError: 
                    pass
    return orig_to_internal

def load_scores(csv_path, orig_to_internal, proxy_col, oracle_col):
    """读取指定表 (如 product.csv / review.csv) 的 Proxy 和 Oracle 概率"""
    proxy_scores, oracle_scores = {}, {}
    if not os.path.exists(csv_path): 
        return proxy_scores, oracle_scores

    with open(csv_path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            orig_id = row.get("id:ID", "").strip()
            if orig_id in orig_to_internal:
                internal_id = orig_to_internal[orig_id]
                try:
                    proxy_scores[internal_id] = float(row[proxy_col])
                    oracle_scores[internal_id] = float(row[oracle_col])
                except Exception: 
                    pass
    return proxy_scores, oracle_scores

def main():
    parser = argparse.ArgumentParser(description="Batch Run Query-Specific AQP Pipeline")
    parser.add_argument("--parent_dataset", default="amazon_data", help="Parent dataset directory name")
    parser.add_argument("--dataset", required=True, help="Dataset folder name (e.g., amazon_extend)")
    parser.add_argument("--fastest_bin", required=True, help="Path to Fastest C++ binary")
    parser.add_argument("--ablation_csv", required=True, help="Path to allocation_strategy_comparison_ablation_sum.csv")
    parser.add_argument("--table1", required=True, help="Table 1 name (e.g., product)")
    parser.add_argument("--table1_proxy", required=True)
    parser.add_argument("--table1_oracle", required=True)
    parser.add_argument("--table2", required=True, help="Table 2 name (e.g., review)")
    parser.add_argument("--table2_proxy", required=True)
    parser.add_argument("--table2_oracle", required=True)
    parser.add_argument("--sum_col", default="price", help="Column name for SUM aggregation")
    parser.add_argument("--sum_label", type=int, default=12, help="Label index for SUM aggregation")
    args = parser.parse_args()

    # 1. 基础路径配置
    base_dir = f"/home/wangshuo/resource/datasets/{args.parent_dataset}/{args.dataset}"
    graph_path = os.path.join(base_dir, "data_graph", "parler.graph")
    graph_bak_path = os.path.join(base_dir, "data_graph", "parler.graph.bak")
    mapping_path = os.path.join(base_dir, "data_graph", "id_mapping.csv")
    config_path = os.path.join(base_dir, "data_graph", "core_nodes_config.json")
    temp_ans_path = os.path.join(base_dir, "ground_truth", "temp_ans.txt")
    out_csv = os.path.join(base_dir, "results", "efficiency", "AQP_Cascade_results.csv")

    # 2. 读取核心节点配置
    print("[*] 正在加载 core_nodes_config.json...")
    if not os.path.exists(config_path):
        print(f"[Error] 找不到配置文件: {config_path}")
        return
    with open(config_path, 'r') as f:
        core_config = json.load(f)

    # 3. 提取 Query-specific Budgets
    print("[*] 提取 Query-specific Budgets (frac=0.1)...")
    query_budgets = load_query_budgets(args.ablation_csv, target_frac=0.1)
    print(f"[+] 成功提取到 {len(query_budgets)} 个查询的专属预算")

    # 4. 加载原始数据图与 ML 概率分数
    print("[*] 加载物理图结构与概率分数...")
    orig_to_internal = load_id_mapping(mapping_path)
    p1_scores, o1_scores = load_scores(os.path.join(base_dir, "csv_data", f"{args.table1}.csv"), orig_to_internal, args.table1_proxy, args.table1_oracle)
    p2_scores, o2_scores = load_scores(os.path.join(base_dir, "csv_data", f"{args.table2}.csv"), orig_to_internal, args.table2_proxy, args.table2_oracle)

    header = ""
    vertices = {}
    edges = []
    with open(graph_path, 'r') as f:
        for line in f:
            if line.startswith('t'): header = line.strip()
            elif line.startswith('v'): 
                p = line.split()
                vertices[int(p[1])] = (int(p[2]), int(p[3]))
            elif line.startswith('e'): edges.append(line.strip())

    # 安全备份原始图
    if not os.path.exists(graph_bak_path):
        os.rename(graph_path, graph_bak_path)
        print("[+] 原始图数据已安全备份为 parler.graph.bak")

    results = []

    try:
        # 5. 逐 Query 自动化运行
        for q_idx, (q_name, budget) in enumerate(query_budgets.items(), 1):
            print(f"\n>>> [{q_idx}/{len(query_budgets)}] 正在处理 {q_name} (Budget B = {budget})")
            
            # 5.1 判断当前 Query 包含哪些表 (根据原始 Label 范围判断)
            has_table1 = False
            has_table2 = False

            if q_name in core_config:
                raw_labels = {int(k) for k in core_config[q_name].keys()}
                # 假设 Amazon 表1(Product)原始Label包含 10..19 或 12；表2(Review)原始Label包含 30..39 或 30
                for l in raw_labels:
                    if l < 20 or l in [1, 2, 10, 11, 12, 13, 14]:
                        has_table1 = True
                    if l >= 20 or l in [3, 4, 5, 30, 31, 32, 33, 34]:
                        has_table2 = True
            else:
                has_table1, has_table2 = True, True

            print(f"    --> 作用的数据表: Table1({args.table1})={has_table1}, Table2({args.table2})={has_table2}")

            # 5.2 为当前 Query 专属裁剪图
            uncertain_nodes = []
            rejected_nodes = set()
            
            for vid in range(len(vertices)):
                # 判断当前节点所属的表，并获取分数
                p_score, o_score = None, None
                
                if vid in p1_scores and has_table1:
                    p_score = p1_scores[vid]
                    o_score = o1_scores.get(vid, 0.0)
                elif vid in p2_scores and has_table2:
                    p_score = p2_scores[vid]
                    o_score = o2_scores.get(vid, 0.0)

                # 如果节点属于当前 Query 激活的表，进行硬裁剪判断
                if p_score is not None:
                    if p_score < 0.4:
                        rejected_nodes.add(vid)  # 硬剔除 (<0.4)
                    elif p_score > 0.6:
                        pass  # 硬接受 (>0.6)
                    else:
                        uncertain_nodes.append((vid, o_score, p_score)) # 灰色地带

            # 打乱灰色地带节点
            random.shuffle(uncertain_nodes)
            
            budget_used = 0
            for vid, o_score, p_score in uncertain_nodes:
                if budget_used < budget:
                    # 在 Oracle 预算内验证真值
                    if o_score <= 0.5:
                        rejected_nodes.add(vid)
                    budget_used += 1
                else:
                    # 预算耗尽，退化为 Proxy 判断
                    if p_score <= 0.5:
                        rejected_nodes.add(vid)

            # print(f"    --> 裁剪完成! 消耗 Oracle 预算: {budget_used}/{budget}. 剔除节点数: {len(rejected_nodes)} (其中硬剔除数: {len(rejected_nodes) - (budget_used - sum(1 for v, o, p in uncertain_nodes[:budget_used] if o > 0.5))})")

            # 统计分别剔除了多少 Product 和 Review
            rejected_t1 = sum(1 for vid in rejected_nodes if vid in p1_scores)
            rejected_t2 = sum(1 for vid in rejected_nodes if vid in p2_scores)

            print(f"    --> 裁剪完成! 消耗 Oracle 预算: {budget_used}/{budget}. 总剔除数: {len(rejected_nodes)} (Product剔除: {rejected_t1}, Review剔除: {rejected_t2})")

            # 覆写当前用于运行的 parler.graph
            with open(graph_path, 'w') as f:
                f.write(header + "\n")
                for vid in range(len(vertices)):
                    label, deg = vertices[vid]
                    if vid in rejected_nodes:
                        label = -1
                    f.write(f"v {vid} {label} {deg}\n")
                for e in edges:
                    f.write(e + "\n")

            # 5.3 写入单 Query 的临时答案文件
            with open(temp_ans_path, 'w') as f:
                f.write(f"{q_name} 1.0 1\n")

            # 5.4 调用 C++ 进行纯结构估计
            cmd = [
                args.fastest_bin, 
                "-d", args.dataset,
                "-q", temp_ans_path,
                "--PARENT_DATASET", args.parent_dataset,
                "--ROOT_LABEL", "1",
                "--SAMPLE_BUDGET", "60000",
                "--AGG_FUNC", "sum",
                "--SUM_TABLE", args.table1,
                "--SUM_COL", args.sum_col,
                "--SUM_LABEL", str(args.sum_label)
            ]
            
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            stdout, stderr = process.communicate()

            # 5.5 从终端输出提取估算结果
            est_value = None
            match = re.search(r"Global Estimated Value:\s+([\d\.]+)", stdout)
            if match:
                est_value = float(match.group(1))
            else:
                match2 = re.search(r"\[Result\]\s+Est\s*:\s*([\d\.]+)", stdout)
                if match2: 
                    est_value = float(match2.group(1))

            if est_value is not None:
                print(f"    --> AQP 估计值: {est_value}")
                results.append({"query_basename": q_name, "T_hat_aqp": est_value})
            else:
                print(f"    --> [Error] 提取估算结果失败！错误输出: {stderr[:200]}")

    finally:
        # 6. 安全恢复原始数据图
        print("\n[*] 正在恢复原始 parler.graph 文件...")
        if os.path.exists(graph_path): 
            os.remove(graph_path)
        if os.path.exists(graph_bak_path):
            os.rename(graph_bak_path, graph_path)
            print("[+] 原图已成功还原！")
        
        # 7. 导出最终的 AQP 估计结果 CSV
        if results:
            keys = results[0].keys()
            os.makedirs(os.path.dirname(out_csv), exist_ok=True)
            with open(out_csv, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=keys)
                writer.writeheader()
                writer.writerows(results)
            print(f"\n✅ 批量 AQP 实验完成！结果已导出至: {out_csv}")

if __name__ == "__main__":
    main()

1. 没有 \Psi 语义投影空间的双阈值代理过滤AQP 

python precision_submatching.py \
  --parent_dataset amazon_data \
  --dataset amazon_extend \
  --fastest_bin /home/wangshuo/projects/FaSTest-main/build/Fastest \
  --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --table1 product \
  --table1_proxy ML3_proxy2_probability \
  --table1_oracle ML3_oracle2_probability \
  --table2 review \
  --table2_proxy ML2_proxy2_probability \
  --table2_oracle ML2_oracle1_probability \
  --sum_col price \
  --sum_label 12 \
  --t1_low 0.2 --t1_high 0.3 \
  --t2_low 0.2 --t2_high 0.3 \
  --num_workers 8

In [17]:
import json
import numpy as np
import pandas as pd

# ==========================================
# 1. 文件路径配置（请按需修改路径）
# ==========================================
# 替换为你的 AQP_Cacade_results.csv 的实际路径
csv_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/AQP_Cascade_results.csv"  
gt_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json"

# ==========================================
# 2. 读取数据
# ==========================================
# 2.1 读取 CSV 估计数据
df_est = pd.read_csv(csv_file)

# 2.2 读取 JSON Ground Truth 数据
with open(gt_file, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# ==========================================
# 3. 数据对齐与误差计算
# ==========================================
records = []
epsilon = 1e-9  # 防除零保护

for _, row in df_est.iterrows():
    raw_key = str(row["query_basename"]).strip()
    t_hat = float(row["T_hat_aqp"])
    
    # 统一清洗 key（去除可能存在的 .graph 后缀，防止对齐失败）
    clean_key = raw_key.replace(".graph", "")
    
    # 寻找 ground truth 中对应的真实值（尝试完整 key、带 .graph 后缀和不带后缀三种情况）
    t_true = gt_data.get(raw_key)
    if t_true is None:
        t_true = gt_data.get(f"{clean_key}.graph")
    if t_true is None:
        t_true = gt_data.get(clean_key)
        
    # 过滤无效值（仅对比两边都有且 T_true > 0 的查询）
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed Relative Error): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (Absolute Relative Error, ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": clean_key,
            "T_hat_aqp (估计值)": t_hat,
            "T_true (真实值)": t_true,
            "Signed_RE (带符号误差)": signed_re,
            "ARE (绝对误差)": are
        })

df = pd.DataFrame(records)

# ==========================================
# 4. 汇总统计与输出
# ==========================================
if df.empty:
    print("❌ 未成功匹配到任何查询，请检查 CSV 中的 query_basename 与 JSON 文件中的 key 名称是否一致！")
else:
    mean_signed_re = df["Signed_RE (带符号误差)"].mean()
    mean_are = df["ARE (绝对误差)"].mean()
    
    # 各种分位数计算 (P50/中位数, P70, P90, P95)
    p50_are = df["ARE (绝对误差)"].median()  # 等价于 .quantile(0.50)
    p70_are = df["ARE (绝对误差)"].quantile(0.70)
    p90_are = df["ARE (绝对误差)"].quantile(0.90)
    p95_are = df["ARE (绝对误差)"].quantile(0.95)

    print("=" * 65)
    print(f"📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 {len(df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)
    
    # 格式化百分比显示详细表格
    df_display = df.copy()
    df_display["Signed_RE (带符号误差)"] = df_display["Signed_RE (带符号误差)"].map("{:.2%}".format)
    df_display["ARE (绝对误差)"] = df_display["ARE (绝对误差)"].map("{:.2%}".format)
    
    # 兼容 Jupyter display 和 终端 print
    try:
        display(df_display.head(10))
    except NameError:
        print(df_display.head(10).to_string(index=False))

📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 176 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : -0.2710  (-27.10%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.4123  (41.23%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.3100  (31.00%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.5738  (57.38%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.8839  (88.39%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.9389  (93.89%)


,query,T_hat_aqp (估计值),T_true (真实值),Signed_RE (带符号误差),ARE (绝对误差)
0,query_5_12,559.62,1202.53,-53.46%,53.46%
1,query_5_117,795.86,754.83,5.44%,5.44%
2,query_5_101,330.44,1147.20,-71.20%,71.20%
3,query_5_113,320.64,225.31,42.31%,42.31%
4,query_5_2,268.58,792.72,-66.12%,66.12%
5,query_5_11,158.49,657.40,-75.89%,75.89%
6,query_5_116,691.68,1180.76,-41.42%,41.42%
7,query_5_120,634.69,503.95,25.94%,25.94%
8,query_5_33,161.87,1291.29,-87.46%,87.46%
9,query_5_37,253.67,866.33,-70.72%,70.72%


2. 基于 \PSi 的双阈值截断, 双阈值根据GT选择F1分数最高的 Proxy 阈值, 效果肯定 >= SUPG/ScaleDoc的阈值, 然后同时引入灰色区间Oracle介入,进一步提高准确性
上面1,2基线的作用是突出我\Psi 语义投影空间的作用, 以及我方法的无偏性和低方差.

In [22]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径配置（根据您的实际文件名修改 csv_name）
# ==========================================
dataset_dir = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results"

# AQP 结果 CSV 路径 (如 AQP_Cascade_results.csv 或 Core_Double_Truncation_amazon_sum.csv)
csv_path = os.path.join(dataset_dir, "efficiency", "Core_Double_Truncation_amazon_sum.csv")

# Ground Truth JSON 路径
gt_json_path = os.path.join(dataset_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")

# ==========================================
# 2. 数据读取与自动对齐
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到估计结果 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取 Ground Truth 并归一化键名
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None}

# 读取 AQP 估计结果 CSV
df = pd.read_csv(csv_path)

# 自动识别估计值列名
est_col = None
for col in ["T_hat_aqp", "T_hat_double_truncation", "T_hat", "estimate"]:
    if col in df.columns:
        est_col = col
        break

if not est_col:
    raise ValueError(f"❌ CSV 中未找到估计值列！当前列名: {df.columns.tolist()}")

# ==========================================
# 3. 逐查询匹配与误差计算
# ==========================================
records = []
epsilon = 1e-9

for _, row in df.iterrows():
    q_name = str(row["query_basename"]).replace(".graph", "").replace(".csv", "")
    t_hat = float(row[est_col])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 {len(eval_df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)

📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 180 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : 0.0264  (2.64%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.1247  (12.47%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.0747  (7.47%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.1122  (11.22%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.2411  (24.11%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.3295  (32.95%)


3. 另一个基线就是, abae论文也用到我的核心实例集上,叫 projection-abaze吧, 也算是我得一个方法吧, 下面是论文内容, 就说先将每个核心实例按照代理分数排序, 通过 pilot 和  简单分层采样(层内是均匀采样)  估计avg. 帮我实现一个新的py文件作为基线:这个基线的作用是突出我重要性采样的作用以及无pilot,闭式分配的作用

In [1]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径与目标配置 (Parler-E 数据集)
# ==========================================
parent_dataset = "amazon_data"
dataset_name = "amazon_extend"

base_results_dir = f"/home/wangshuo/resource/datasets/{parent_dataset}/{dataset_name}/results"

# 估计结果 CSV 路径
csv_path = os.path.join(base_results_dir, "efficiency", "Projection_ABae_amazon_sum.csv")
# 兜底路径（如果上面找不到，尝试默认文件名）
if not os.path.exists(csv_path):
    csv_path = os.path.join(base_results_dir, "efficiency", "Projection_ABae_results_sum.csv")

# Ground Truth JSON 路径 (Parler SUM)
gt_json_path = os.path.join(base_results_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")

# ==========================================
# 2. 读取与预处理
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到估计结果 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取真值 JSON 并归一化键名 (去 .graph)
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

# 读取 Projection-ABae 结果 CSV
df = pd.read_csv(csv_path)

# 自动匹配估计值列名
est_col = None
for col in ["T_hat_abae", "T_hat_abae_avg", "T_hat", "T_hat_sum"]:
    if col in df.columns:
        est_col = col
        break

if not est_col:
    raise ValueError(f"❌ CSV 中未找到估计值列！当前列名: {df.columns.tolist()}")

# 3) 规范化 query_basename 键名
df["query_clean"] = df["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)

# 如果存在多轮 run_id，按 query 对估计值 T_hat 求均值
if "run_id" in df.columns:
    df_eval = df.groupby("query_clean")[est_col].mean().reset_index()
else:
    df_eval = df[["query_clean", est_col]].copy()

# ==========================================
# 3. 对齐真值与计算误差
# ==========================================
records = []
epsilon = 1e-9

for _, row in df_eval.iterrows():
    q_name = row["query_clean"]
    t_hat = float(row[est_col])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 Projection-ABae 方法评估结果汇总 (成功对齐 {len(eval_df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)



📊 Projection-ABae 方法评估结果汇总 (成功对齐 180 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : -0.1018  (-10.18%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.1923  (19.23%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.0925  (9.25%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.1550  (15.50%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.5106  (51.06%)
6. 绝对值相对误差 P95 (P95 ARE)         : 1.0000  (100.00%)


4. Our Proxy 算法的输出结果

In [3]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径与目标配置
# ==========================================
# dataset_dir = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results"
dataset_dir = "/home/wangshuo/resource/datasets/parler_data/dataset_three/results"
# 优先读取消融 CSV 文件，若不存在则读取常规 CSV
csv_path = os.path.join(dataset_dir, "efficiency", "allocation_strategy_comparison_count_ADV.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(dataset_dir, "efficiency", "allocation_strategy_comparison_count_ADV.csv")

# Ground Truth JSON 路径
# gt_json_path = os.path.join(dataset_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
gt_json_path = os.path.join(dataset_dir, "T_true_ML1_oracle2_probability_ML2_oracle2_probability_count.json")

TARGET_METHOD = "8_POSSA"  # 目标方法（兼容 8_POSSA 或 POSS）
TARGET_FRAC = 0.1          # 目标预算比例

# ==========================================
# 2. 数据读取与预处理
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取 Ground Truth JSON 并归一化键名
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None}

# 读取 CSV 文件
df = pd.read_csv(csv_path)

# 1) 过滤方法 (8_POSSA 或 POSS)
df_filtered = df[df["method"].isin(["8_POSSA", "POSS"])].copy()

# 2) 过滤采样率 (budget_frac == 0.1)
if "budget_frac" in df_filtered.columns:
    df_filtered = df_filtered[np.isclose(df_filtered["budget_frac"], TARGET_FRAC, atol=1e-4)].copy()

if df_filtered.empty:
    raise ValueError(f"❌ 未找到符合 method={TARGET_METHOD} 且 budget_frac={TARGET_FRAC} 的记录！")

# 3) 规范化 query_basename 键名
df_filtered["query_clean"] = df_filtered["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)

# ==========================================
# 3. 对齐真值与计算误差 (完全按脚本 1 粒度：每条运行记录独立计算)
# ==========================================
records = []
epsilon = 1e-9

# 直接遍历每一条运行记录（不提前 groupby 求 T_hat 平均）
for _, row in df_filtered.iterrows():
    q_name = row["query_clean"]
    t_hat = float(row["T_hat"])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "run_id": row.get("run_id", 1),
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 8_POSSA (POSS) 方法评估结果汇总 (完全对齐脚本 1 粒度)")
    print(f"   数据文件: {os.path.basename(csv_path)} | 匹配记录数: {len(eval_df)} 条")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)

📊 8_POSSA (POSS) 方法评估结果汇总 (完全对齐脚本 1 粒度)
   数据文件: allocation_strategy_comparison_count_ADV.csv | 匹配记录数: 1230 条
1. 带符号相对误差均值 (Mean Signed RE) : -0.0022  (-0.22%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.1895  (18.95%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.1483  (14.83%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.2368  (23.68%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.3917  (39.17%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.4908  (49.08%)
